# gsplat 1.5.3 and 1.6.0 on a Colab T4

This notebook runs the same experiment as `run_scene.bat` on a free Colab T4 GPU: one training run of the official gsplat trainer, one evaluation pass per saved checkpoint, then the summary table from `compare.py`.

It installs one gsplat version per session, in its own virtual environment, so Colab's own packages are never touched. Version 1.5.3 uses gsplat's official prebuilt wheel. Version 1.6.0 has no wheel, so it is compiled from the pinned commit, which takes a long time on the 2 CPU cores of a Colab runtime.

**How to connect in VS Code**

1. Install the official Google Colab extension and open this notebook.
2. Click Select Kernel, then Colab, then New Colab Server.
3. Choose a GPU runtime and pick T4.
4. Run the cells from top to bottom. Only the Settings cell needs editing.

**One version per session.** To switch between 1.5.3 and 1.6.0, restart the runtime (disconnect, then connect to a new server) and run again with the other `VERSION`.

**Using a RealityScan scene.** Those scenes are made on the lab PC and are not in the repo. In the VS Code Explorer, right click the folder `data\<scene>` (for example `data\stump_rs`) and choose Upload to Colab. It lands at `/content/<scene>`, and the Data cell links it into place.

**Results are temporary.** The runtime disk is wiped when the session ends. To keep results, mount Google Drive before running the Drive cell, or copy the results off before you disconnect (see the last cell).

## 1. Settings

This is the only cell you edit. `VERSION` is `"153"` or `"160"`. `LENGTH` is `"30k"` for the official full run, or any number of thousands of steps such as `"7k"` or `"10k"`. `SCENE` is `"stump"` or the name of an uploaded scene. `MCMC` switches to the official mcmc benchmark (at most `CAP_MAX` Gaussians). `PACKED` uses the trainer's own `--packed` flag, which needs less GPU memory and is slightly slower.

The constants below the divider are the same values the lab PC setup uses. The two commits are the pinned gsplat versions from `setup_env.py`.

In [ ]:
import os

# ---- Edit these ----------------------------------------------------------
VERSION = "153"      # "153" for gsplat 1.5.3 or "160" for gsplat 1.6.0
LENGTH = "7k"        # "30k" for the official full run, or any number of thousands, such as "7k" or "10k"
SCENE = "stump"      # "stump", or the name of a scene folder uploaded to /content
MCMC = False         # True: official mcmc benchmark, at most CAP_MAX Gaussians
PACKED = False       # True: the trainer's --packed flag, less GPU memory and slightly slower
REPO_URL = "https://github.com/Lawrensu/gsplat-experiment.git"




# ---- Constants (leave as they are) ---------------------------------------
CAP_MAX = 1000000    # the CAP_MAX value in the official mcmc.sh

GSPLAT_URL = "https://github.com/nerfstudio-project/gsplat.git"
GSPLAT_153_SHA = "937e29912570c372bed6747a5c9bf85fed877bae"
GSPLAT_160_SHA = "512d366b67073d77ca099ede742683c165dfc23b"

FULL_LENGTH = "30k"
LENGTH_PATTERN = r"^[1-9][0-9]*k$"
SCENE_PATTERN = r"^[A-Za-z0-9][A-Za-z0-9_.-]*$"
STEPS_PER_K = 1000

CONTENT_DIR = "/content"
REPO_DIR = "/content/gsplat-experiment"
DRIVE_DIR = "/content/drive/MyDrive"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/gsplat-experiment/results"
LOCAL_RESULTS_DIR = "/content/gsplat-experiment/results"

STUMP_SCENE = "stump"
STUMP_DATA_REL = "data/360_v2/stump"
STUMP_FACTOR = 4
OTHER_DATA_REL = "data/{scene}/colmap"
OTHER_FACTOR = 1

GUARDED_TRAIN_REL = "guarded_train.py"
COMPARE_REL = "compare.py"
TRAINER = "simple_trainer.py"
CUDA_VISIBLE_DEVICES = "0"
MEMORY_CAP_EXIT_CODE = 2

CUDA_HOME = "/usr/local/cuda"
TORCH_CUDA_ARCH_LIST = "7.5"     # T4 only, which shortens the 1.6.0 build
BUILD_EXPERIMENTAL = "0"         # same as the lab PC: skip the experimental kernels
MAX_JOBS = os.cpu_count() or 2

# Keeps live output readable: unbuffered Python, and tqdm progress bars that update every few seconds
STREAM_ENV = {"PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "10"}

# Entries of examples/requirements.txt that simple_trainer.py only needs for options the
# COLMAP baseline does not use. Only these may be dropped when the requirements install fails
OPTIONAL_REQS = {
    "nvidia-ncore": "NCore dataset example only",
    "ppisp": "only --post_processing ppisp",
    "fused-bilagrid": "only the fused bilateral grid option",
}

VERSIONS = {
    "153": {
        "label": "1.5.3",
        "sha": GSPLAT_153_SHA,
        "python": "3.10",
        "venv": "/content/venv153",
        "clone": "/content/gsplat-1.5.3",
        "torch_index": "https://download.pytorch.org/whl/cu124",
        "torch_packages": ["torch==2.4.1", "torchvision==0.19.1"],
        # gsplat's own dependencies, plus the build tools the examples requirements need
        "base_packages": ["setuptools", "wheel", "ninja", "numpy<2.0.0", "jaxtyping", "rich", "typing_extensions"],
        "wheel_index": "https://docs.gsplat.studio/whl/pt24cu124",
        "wheel_requirement": "gsplat==1.5.3",
        "from_source": False,
    },
    "160": {
        "label": "1.6.0",
        "sha": GSPLAT_160_SHA,
        "python": "3.11",
        "venv": "/content/venv160",
        "clone": "/content/gsplat-1.6.0",
        "torch_index": "https://download.pytorch.org/whl/cu126",
        "torch_packages": ["torch==2.9.1", "torchvision==0.24.1"],
        "base_packages": ["setuptools>=64", "wheel", "ninja", "rich", "numpy>=2,<3"],
        "wheel_index": None,
        "wheel_requirement": None,
        "from_source": True,
    },
}
GLM_SUBMODULE = "gsplat/cuda/csrc/third_party/glm"

## 2. Helpers

Shared functions used by the cells below. `run` streams a command's output line by line and returns its exit code. The command builders follow `run_scene.bat` exactly, so a run here is the same run as on the lab PC. The last line checks your settings and stops early with a clear message if one is wrong.

In [ ]:
import glob
import posixpath
import re
import shlex
import shutil
import subprocess
import sys




def run(cmd, cwd=None, env=None, log_path=None):
    """Run a command and stream its output live, line by line.

    Args:
        cmd: Command as a list of arguments.
        cwd: Working directory, or None for the current one.
        env: Extra environment variables, added to the current environment.
        log_path: Text file to append the output to, or None for no log file.

    Returns:
        The exit code of the command.
    """
    full_env = dict(os.environ)
    full_env.update(STREAM_ENV)
    if env:
        full_env.update(env)
    cmd = [str(part) for part in cmd]
    header = "$ " + " ".join(shlex.quote(part) for part in cmd)
    print(header, flush=True)
    log_file = None
    if log_path:
        os.makedirs(posixpath.dirname(log_path), exist_ok=True)
        log_file = open(log_path, "a", encoding="utf-8", buffering=1)
        log_file.write(header + "\n")
    try:
        process = subprocess.Popen(
            cmd, cwd=cwd, env=full_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            if log_file:
                log_file.write(line)
        return process.wait()
    finally:
        if log_file:
            log_file.close()




def require(exit_code, what):
    """Stop the notebook with a clear error when a command failed.

    Args:
        exit_code: Exit code returned by run.
        what: Short description of the step, used in the error message.

    Returns:
        None
    """
    if exit_code != 0:
        raise RuntimeError(f"{what} failed with exit code {exit_code}. See the output above.")




def validate_settings(version, length, scene):
    """Check the settings and raise a clear error when one is invalid.

    The length rule is the one run_scene.bat uses: 30k, or a number of thousands with k.

    Args:
        version: "153" or "160".
        length: "30k", or another number of thousands such as "7k".
        scene: "stump" or an uploaded scene folder name.

    Returns:
        None
    """
    if version not in VERSIONS:
        raise ValueError(f'VERSION must be one of {sorted(VERSIONS)}, got "{version}"')
    if length != FULL_LENGTH and not re.match(LENGTH_PATTERN, length):
        raise ValueError(
            f'LENGTH must be "{FULL_LENGTH}" or a whole number of thousands such as "7k" or "10k", got "{length}"'
        )
    if not re.match(SCENE_PATTERN, scene):
        raise ValueError(f'SCENE must be a folder name such as "stump" or "amorphophallus_rs", got "{scene}"')




def length_args(length):
    """Trainer arguments that shorten the run, as in run_scene.bat.

    Args:
        length: "30k" for the full run, or another number of thousands such as "7k".

    Returns:
        An empty list for 30k, else --max_steps and --save_steps with the step count.
    """
    if length == FULL_LENGTH:
        return []
    steps = str(int(length[:-1]) * STEPS_PER_K)
    return ["--max_steps", steps, "--save_steps", steps]




def option_args(mcmc, packed):
    """Trainer arguments for the mcmc and packed options, in the order run_scene.bat uses.

    Args:
        mcmc: True for the mcmc benchmark (adds the Gaussian cap).
        packed: True to add the trainer's --packed flag.

    Returns:
        The list of arguments, possibly empty.
    """
    args = []
    if mcmc:
        args += ["--strategy.cap-max", str(CAP_MAX)]
    if packed:
        args += ["--packed"]
    return args




def config_name(mcmc):
    """Name of the trainer config.

    Args:
        mcmc: True for the mcmc benchmark.

    Returns:
        "mcmc" or "default".
    """
    return "mcmc" if mcmc else "default"




def run_name(length, mcmc, packed):
    """Name of the run folder, for example 7k, 30k_mcmc or 10k_mcmc_packed.

    Args:
        length: The LENGTH setting.
        mcmc: True for the mcmc benchmark.
        packed: True for the packed option.

    Returns:
        The run folder name.
    """
    return length + ("_mcmc" if mcmc else "") + ("_packed" if packed else "")




def scene_data(scene):
    """Dataset folder and image downscale factor for a scene, as in run_scene.bat.

    Args:
        scene: "stump" or an uploaded scene folder name.

    Returns:
        A tuple (data folder, data factor).
    """
    if scene == STUMP_SCENE:
        return posixpath.join(REPO_DIR, STUMP_DATA_REL), STUMP_FACTOR
    return posixpath.join(REPO_DIR, OTHER_DATA_REL.format(scene=scene)), OTHER_FACTOR




def result_dir(results_root, version, length, mcmc, packed, scene):
    """Folder where one run writes its results.

    Args:
        results_root: Folder that holds the results of all runs.
        version: "153" or "160".
        length: The LENGTH setting.
        mcmc: True for the mcmc benchmark.
        packed: True for the packed option.
        scene: The scene name.

    Returns:
        results_root/<version>/<length>[_mcmc][_packed]/<scene>
    """
    return posixpath.join(results_root, version, run_name(length, mcmc, packed), scene)




def common_args(data_dir, factor, mcmc, packed, res_dir):
    """The COMMON arguments of run_scene.bat, used for both training and evaluation.

    Args:
        data_dir: Dataset folder.
        factor: Image downscale factor.
        mcmc: True for the mcmc benchmark.
        packed: True for the packed option.
        res_dir: Result folder of the run.

    Returns:
        The list of arguments.
    """
    return [
        "--disable_viewer", "--data_factor", str(factor), *option_args(mcmc, packed),
        "--render_traj_path", "ellipse", "--data_dir", data_dir, "--result_dir", res_dir,
    ]




def train_command(python, gpu_log, config, common, extra):
    """Training command: the trainer through guarded_train.py, without eval, saving ply files.

    Args:
        python: The venv python.
        gpu_log: Path of the GPU log csv.
        config: Trainer config name.
        common: Result of common_args.
        extra: Result of length_args.

    Returns:
        The command as a list of arguments.
    """
    guarded = posixpath.join(REPO_DIR, GUARDED_TRAIN_REL)
    return [python, guarded, gpu_log, TRAINER, config, "--eval_steps", "-1", "--save_ply", *common, *extra]




def eval_command(python, gpu_log, config, common, ckpt):
    """Evaluation command for one checkpoint.

    Args:
        python: The venv python.
        gpu_log: Path of the GPU log csv.
        config: Trainer config name.
        common: Result of common_args.
        ckpt: Checkpoint file to evaluate.

    Returns:
        The command as a list of arguments.
    """
    guarded = posixpath.join(REPO_DIR, GUARDED_TRAIN_REL)
    return [python, guarded, gpu_log, TRAINER, config, *common, "--ckpt", ckpt]




validate_settings(VERSION, LENGTH, SCENE)
print(f"Settings OK: gsplat {VERSIONS[VERSION]['label']}, {LENGTH}, scene {SCENE}, "
      f"mcmc={MCMC}, packed={PACKED}")

## 3. GPU check

Shows the GPU and the CUDA toolkit. If there is no GPU, the notebook stops here. In that case, open Select Kernel, choose Colab, create a new server and pick a GPU runtime (T4).

In [ ]:
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is None:
    raise RuntimeError("No GPU found (nvidia-smi is missing). Pick a GPU runtime: Select Kernel, Colab, New Colab Server, then T4.")
require(run([nvidia_smi]), "nvidia-smi (is a GPU runtime selected?)")

nvcc = shutil.which("nvcc") or posixpath.join(CUDA_HOME, "bin", "nvcc")
if os.path.exists(nvcc):
    run([nvcc, "--version"])
else:
    print(f"warning: nvcc not found at {nvcc}. gsplat 1.5.3 does not need it, 1.6.0 does.")

## 4. Google Drive (optional)

If you want the results kept after the session ends, run the extension command "Colab: Mount Google Drive to Server..." from the command palette first, then run this cell. Without Drive the results go to the runtime disk and are lost when the session ends.

In [ ]:
if os.path.isdir(DRIVE_DIR):
    RESULTS_ROOT = DRIVE_RESULTS_DIR
    print(f"Google Drive is mounted. Results go to {RESULTS_ROOT}")
else:
    RESULTS_ROOT = LOCAL_RESULTS_DIR
    print(f"Google Drive is not mounted. Results go to {RESULTS_ROOT} and are lost when the session ends")

## 5. Get the experiment repo

Clones the experiment repo (scripts and the stump scene) to `/content/gsplat-experiment`, or updates it if it is already there.

In [ ]:
if os.path.isdir(posixpath.join(REPO_DIR, ".git")):
    require(run(["git", "-C", REPO_DIR, "pull", "--ff-only"]), "git pull")
else:
    require(run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR]), "git clone")
for name in (GUARDED_TRAIN_REL, COMPARE_REL):
    if not os.path.isfile(posixpath.join(REPO_DIR, name)):
        raise FileNotFoundError(f"{name} is missing in {REPO_DIR}. Check REPO_URL.")

## 6. Set up the chosen gsplat version

Creates a virtual environment with uv and installs everything into it. Training later runs with that environment's Python, so Colab's own packages stay as they are.

1. Version 1.5.3: torch 2.4.1 (CUDA 12.4) and gsplat's dependencies, then gsplat's official prebuilt wheel. Nothing is compiled.
2. Version 1.6.0: torch 2.9.1 (CUDA 12.6), then gsplat is compiled from the pinned commit for the T4. This takes a long time on 2 CPU cores, often 20 minutes or more, and the output can look stuck. Let it run.
3. Both: gsplat is cloned at the pinned commit (the trainer script comes from it), then the examples requirements are installed. If that fails, it retries once without the optional packages and says which were skipped.
4. Last, a tiny rasterization runs on the GPU, so a missing CUDA kernel shows up now and not in the middle of training.

If you run the cell again in the same session, the install is skipped and only the check runs.

In [ ]:
def uv_command():
    """Return the uv command, installing uv into the system Python if it is missing.

    Args:
        None

    Returns:
        The command as a list, ready to be followed by uv arguments.
    """
    if shutil.which("uv"):
        return ["uv"]
    require(run([sys.executable, "-m", "pip", "install", "-q", "uv"]), "pip install uv")
    return [sys.executable, "-m", "uv"]




def venv_python(spec):
    """Path of the python inside a version's venv.

    Args:
        spec: One entry of VERSIONS.

    Returns:
        The path of the venv python.
    """
    return posixpath.join(spec["venv"], "bin", "python")




def uv_install(uv, python, args, env=None):
    """Run uv pip install into a venv.

    Args:
        uv: Result of uv_command.
        python: The venv python to install into.
        args: Arguments for uv pip install.
        env: Extra environment variables, or None.

    Returns:
        The exit code.
    """
    return run([*uv, "pip", "install", "--python", python, *args], env=env)




def build_env():
    """Environment for steps that compile CUDA code (the 1.6.0 build and the examples requirements).

    Args:
        None

    Returns:
        A dict of environment variables to add.
    """
    return {
        "CUDA_HOME": CUDA_HOME,
        "PATH": posixpath.join(CUDA_HOME, "bin") + os.pathsep + os.environ.get("PATH", ""),
        "TORCH_CUDA_ARCH_LIST": TORCH_CUDA_ARCH_LIST,
        "BUILD_EXPERIMENTAL": BUILD_EXPERIMENTAL,
        "MAX_JOBS": str(MAX_JOBS),
    }




def clone_at_commit(spec):
    """Clone gsplat at the pinned commit with a shallow fetch.

    The glm submodule is added for the 1.6.0 source build.

    Args:
        spec: One entry of VERSIONS.

    Returns:
        The path of the clone.
    """
    clone, sha = spec["clone"], spec["sha"]
    head = None
    if os.path.isdir(posixpath.join(clone, ".git")):
        head = subprocess.run(["git", "-C", clone, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    if head != sha:
        shutil.rmtree(clone, ignore_errors=True)
        os.makedirs(clone)
        require(run(["git", "init", "-q"], cwd=clone), "git init")
        require(run(["git", "remote", "add", "origin", GSPLAT_URL], cwd=clone), "git remote add")
        require(run(["git", "fetch", "--depth", "1", "origin", sha], cwd=clone), "git fetch of the pinned commit")
        require(run(["git", "checkout", "-q", "FETCH_HEAD"], cwd=clone), "git checkout")
    if spec["from_source"] and not os.path.exists(posixpath.join(clone, GLM_SUBMODULE, "glm", "glm.hpp")):
        code = run(["git", "submodule", "update", "--init", "--depth", "1", GLM_SUBMODULE], cwd=clone)
        if code != 0:
            code = run(["git", "submodule", "update", "--init", GLM_SUBMODULE], cwd=clone)
        require(code, "git submodule update (glm headers)")
    return clone




def requirement_name(line):
    """Normalized project name of one requirements line.

    Args:
        line: A line of a requirements file.

    Returns:
        The project name in lower case with dashes, or None for comments and blank lines.
    """
    text = line.strip()
    if not text or text.startswith(("#", "-")):
        return None
    match = re.match(r"^([A-Za-z0-9][A-Za-z0-9._-]*)(\[[^\]]*\])?\s*(@|[<>=!~;]|$)", text)
    if match and not text.startswith("git+"):
        name = match.group(1)
    else:
        match = re.match(r"^git\+\S*?/([A-Za-z0-9._-]+?)(\.git)?(@\S*)?$", text)
        if not match:
            return None
        name = match.group(1)
    return name.lower().replace("_", "-").replace(".", "-")




def filter_requirements(text, names):
    """Remove the requirements lines of the given projects.

    Args:
        text: Content of a requirements file.
        names: Set of project names to remove.

    Returns:
        A tuple (new text, list of the removed lines).
    """
    kept, dropped = [], []
    for line in text.splitlines(keepends=True):
        if requirement_name(line) in names:
            dropped.append(line.strip())
        else:
            kept.append(line)
    return "".join(kept), dropped




def install_examples_requirements(uv, python, spec, clone):
    """Install examples/requirements.txt, retrying once without the optional packages.

    Args:
        uv: Result of uv_command.
        python: The venv python.
        spec: One entry of VERSIONS.
        clone: Path of the gsplat clone.

    Returns:
        The list of requirement lines that were skipped (empty when none).
    """
    req_path = posixpath.join(clone, "examples", "requirements.txt")
    env = build_env()
    print("Some entries (fused-ssim and similar) compile CUDA code, which takes a few minutes.")
    code = uv_install(uv, python, ["--no-build-isolation", "-r", req_path], env)
    if code == 0:
        return []
    with open(req_path, encoding="utf-8") as handle:
        text = handle.read()
    filtered, dropped = filter_requirements(text, set(OPTIONAL_REQS))
    if not dropped:
        require(code, "installing examples/requirements.txt")
    filtered_path = posixpath.join(spec["venv"], "examples_requirements_filtered.txt")
    with open(filtered_path, "w", encoding="utf-8") as handle:
        handle.write(filtered)
    print("\nThe requirements install failed. Retrying once without the optional packages:")
    for line in dropped:
        print(f"  SKIPPED: {line}   [{OPTIONAL_REQS[requirement_name(line)]}]")
    require(uv_install(uv, python, ["--no-build-isolation", "-r", filtered_path], env),
            "installing examples/requirements.txt without the optional packages")
    return dropped




def install_version(spec):
    """Create the venv and install torch, gsplat and the examples requirements.

    Args:
        spec: One entry of VERSIONS.

    Returns:
        The list of requirement lines that were skipped.
    """
    uv = uv_command()
    python = venv_python(spec)
    shutil.rmtree(spec["venv"], ignore_errors=True)
    require(run([*uv, "venv", spec["venv"], "--python", spec["python"]]), "uv venv")

    print(f"\n== torch {spec['torch_packages'][0].split('==')[1]} from {spec['torch_index']}")
    require(uv_install(uv, python, [*spec["torch_packages"], "--index-url", spec["torch_index"]]), "installing torch")

    print("\n== build tools and gsplat dependencies")
    require(uv_install(uv, python, spec["base_packages"]), "installing the build tools")

    clone = clone_at_commit(spec)
    if spec["from_source"]:
        print("\n== building gsplat from source at the pinned commit")
        print("NOTE: this compiles CUDA code and takes a long time on 2 CPU cores (often 20 minutes or more).")
        print("The output can look stuck. Let it run.")
        build = run([*uv, "pip", "install", "--python", python, "-e", ".", "--no-build-isolation", "-v"],
                    cwd=clone, env=build_env())
        require(build, "building gsplat")
    else:
        print("\n== gsplat prebuilt wheel (no compile)")
        require(uv_install(uv, python, [spec["wheel_requirement"], "--index-url", spec["wheel_index"], "--no-deps"]),
                "installing the gsplat wheel")

    print("\n== examples requirements")
    return install_examples_requirements(uv, python, spec, clone)




VERIFY_CODE = """
import torch, gsplat
from gsplat import rasterization
print("torch", torch.__version__, "cuda", torch.version.cuda, "| gsplat", gsplat.__version__)
print("GPU:", torch.cuda.get_device_name(0))
torch.manual_seed(0)
dev = "cuda"
n = 200
means = torch.randn(n, 3, device=dev) * 0.5
means[:, 2] += 2.0
quats = torch.nn.functional.normalize(torch.randn(n, 4, device=dev), dim=-1)
scales = torch.full((n, 3), 0.05, device=dev)
opacities = torch.full((n,), 0.8, device=dev)
colors = torch.rand(n, 3, device=dev)
viewmats = torch.eye(4, device=dev)[None]
Ks = torch.tensor([[100.0, 0, 32], [0, 100.0, 32], [0, 0, 1.0]], device=dev)[None]
render, alpha, info = rasterization(means, quats, scales, opacities, colors, viewmats, Ks, 64, 64)
assert torch.isfinite(render).all() and float(alpha.sum()) > 0, "rasterization output is empty or not finite"
print("rasterization OK on the GPU, image", tuple(render.shape))
"""




def setup_version(spec):
    """Install the version unless this session already did, then verify it on the GPU.

    Args:
        spec: One entry of VERSIONS.

    Returns:
        The path of the venv python.
    """
    python = venv_python(spec)
    marker = posixpath.join(spec["venv"], f".ready_{spec['sha']}")
    print(f"gsplat {spec['label']} (commit {spec['sha'][:7]}), venv {spec['venv']}")
    for other in VERSIONS.values():
        if other is not spec and os.path.exists(other["venv"]):
            print(f"note: gsplat {other['label']} was also set up in this session. One version per session is the plan.")
    if os.path.exists(marker):
        print("Already set up in this session, skipping the install.")
    else:
        skipped = install_version(spec)
        with open(marker, "w", encoding="utf-8") as handle:
            handle.write("\n".join(skipped) + "\n")
        if skipped:
            print("\nSkipped optional packages: " + ", ".join(skipped))
    print("\n== check on the GPU")
    require(run([python, "-c", VERIFY_CODE], cwd=CONTENT_DIR), "the GPU check of the install")
    return python




SPEC = VERSIONS[VERSION]
PY = setup_version(SPEC)
EXAMPLES_DIR = posixpath.join(SPEC["clone"], "examples")
print(f"\nReady. Python: {PY}\nTrainer folder: {EXAMPLES_DIR}")

## 7. Data

The stump scene comes with the repo (`data/360_v2/stump`, image factor 4). Any other scene is read from `data/<scene>/colmap` with factor 1. If that folder is missing but you uploaded the scene, it is linked into place from `/content/<scene>/colmap`. Otherwise the cell stops and tells you how to upload it.

In [ ]:
DATA_DIR, FACTOR = scene_data(SCENE)

if not os.path.isdir(DATA_DIR) and SCENE != STUMP_SCENE:
    uploaded = posixpath.join(CONTENT_DIR, SCENE, "colmap")
    if os.path.isdir(uploaded):
        os.makedirs(posixpath.dirname(DATA_DIR), exist_ok=True)
        os.symlink(uploaded, DATA_DIR)
        print(f"Linked {uploaded} to {DATA_DIR}")
    else:
        raise FileNotFoundError(
            f"Scene data not found at {DATA_DIR} or {uploaded}.\n"
            f"On the lab PC, right click the folder data\\{SCENE} in the VS Code Explorer and choose "
            f"Upload to Colab, then run this cell again."
        )

if not os.path.isdir(posixpath.join(DATA_DIR, "sparse")):
    raise FileNotFoundError(f"No sparse folder in {DATA_DIR}. The COLMAP reconstruction is missing or incomplete.")
print(f"Scene {SCENE}: data {DATA_DIR}, data factor {FACTOR}")

## 8. Train

The same command as `run_scene.bat`: the official trainer through `guarded_train.py`, which caps GPU memory at 90 percent and logs the GPU to `gpu_log.csv`. Output streams live and is also saved to `train_log.txt` in the results folder. Exit code 2 means the memory cap stopped the run, in which case set `MCMC = True` (or `PACKED = True`) and run again.

In [ ]:
RES = result_dir(RESULTS_ROOT, VERSION, LENGTH, MCMC, PACKED, SCENE)
CONFIG = config_name(MCMC)
COMMON = common_args(DATA_DIR, FACTOR, MCMC, PACKED, RES)
EXTRA = length_args(LENGTH)
GPU_LOG = posixpath.join(RES, "gpu_log.csv")
TRAIN_LOG = posixpath.join(RES, "train_log.txt")

os.makedirs(RES, exist_ok=True)
if os.path.exists(GPU_LOG):
    os.remove(GPU_LOG)

print(f"==== gsplat {SPEC['label']} {SCENE} {run_name(LENGTH, MCMC, PACKED)}: training ====")
exit_code = run(
    train_command(PY, GPU_LOG, CONFIG, COMMON, EXTRA),
    cwd=EXAMPLES_DIR,
    env={"CUDA_VISIBLE_DEVICES": CUDA_VISIBLE_DEVICES},
    log_path=TRAIN_LOG,
)
if exit_code == MEMORY_CAP_EXIT_CODE:
    raise RuntimeError("Training was stopped by the 90 percent GPU memory cap (exit code 2). "
                       "Set MCMC = True in the Settings cell, which caps the number of Gaussians, and run again.")
require(exit_code, "Training")

## 9. Evaluate

Runs the evaluation and the trajectory render once for each saved checkpoint, with the same options as training, as `run_scene.bat` does. The GPU log is appended to. The output is also saved to `eval_log.txt`.

In [ ]:
CKPTS = sorted(glob.glob(posixpath.join(RES, "ckpts", "*")))
if not CKPTS:
    raise FileNotFoundError(f"No checkpoints found in {posixpath.join(RES, 'ckpts')}")

for ckpt in CKPTS:
    print(f"\n==== gsplat {SPEC['label']} {SCENE} {run_name(LENGTH, MCMC, PACKED)}: evaluation of {posixpath.basename(ckpt)} ====")
    exit_code = run(
        eval_command(PY, GPU_LOG, CONFIG, COMMON, ckpt),
        cwd=EXAMPLES_DIR,
        env={"CUDA_VISIBLE_DEVICES": CUDA_VISIBLE_DEVICES},
        log_path=posixpath.join(RES, "eval_log.txt"),
    )
    require(exit_code, f"Evaluation of {posixpath.basename(ckpt)}")

## 10. Summary

First the summary of this run: Gaussians in each saved splat, plus quality, time and memory. Then the table of every run found in the results folder, which compares the two versions on the same scene and length once both have been run.

In [ ]:
compare = posixpath.join(REPO_DIR, COMPARE_REL)
require(run([PY, compare, "--run", RES]), "compare.py --run")
print()
run([PY, compare, RESULTS_ROOT])

## 11. Get the results

Everything is in the results folder printed in the Drive cell. The splat is in `ply/point_cloud_<step>.ply` inside the run folder, and the numbers are in `stats`.

1. If Google Drive was mounted, the results are already in `MyDrive/gsplat-experiment/results`. Download them from Drive.
2. Otherwise, in VS Code open the Colab server's file view (the extension can mount the server's file system in the Explorer), find the run folder under `/content/gsplat-experiment/results`, and download the `ply` files and the `stats` folder before the session ends.

To view a splat, open the SuperSplat editor at https://superspl.at/editor and drag the `.ply` file into it.

Runtime disks are wiped when the session ends, so copy the results off first.